# Infectious disease & Meteorlogical triggers

### Local Data Dictionary:
| Variable Name | Description |
| --- | --- |
|`disease_df`|Raw infectious disease DataFrame|
|`weather_df`|Combined DataFrame with all the weather impacts needed|
|`max_temp_df`, `min_temp_df`, `mean_temp_df`|Raw temperature DataFrames|
|`rainfall_df`|Raw total rainfall DataFrame|
|`humidity_df`|Raw mean humidity DataFrame|

### 'weather_df' Data Dictionary:
| Column | Description | Value |
| --- | --- | --- |

### Problem from dataset:
- (solved)
- 1990/02/19 exists in weather dataframe -> unable to convert data type to datetime (solved by dropping the related rows before doing data type conversion)
- weather statistics gained from the Hong Kong Observatory station cannot gain location (solved by manually coorect the format in the csv file)
- NA values in year, date, day (solved by dropping the related rows)
- value 'Trace' (value less than 0.05mm) in rainfall datasets
- consist incomplete data (no data, and incomplete) (solved by dropping rows with no data, and keep the rest)

---

In [1]:
# Import neccessary libraries
import os
import re 
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# Display setting
pd.options.display.max_rows = 5
pd.options.display.max_columns = 25


## Data Preparation

In [2]:
path = os.getcwd()
path += "\\dataset"
csv_files = glob.glob(os.path.join(path, "*.csv"))
for f in csv_files:
    flux_df = pd.read_csv(f)
flux_df

,Year,Week,From,To,ILI_FMC,ILI_PMP,H1,H3,B,AandB,...,Adm_65_higher,Adm_All,ILI_AED,Fever_CCCKG,Fever_RCHE,ILI_CMP,SevereCase_0_17,SevereCase_18_49,SevereCase_50_64,SevereCase_65_higher
0,2014,1,29/12/2013,04/01/2014,NaN,45.0,93,111,64,268,...,0.404,0.234,198.0,NaN,0.0009,3.86,0.0,1.0,3.0,1.0
1,2014,2,05/01/2014,11/01/2014,NaN,43.8,230,140,109,479,...,0.630,0.347,204.1,0.0088,0.0011,4.90,2.0,2.0,4.0,8.0
2,2014,3,12/01/2014,18/01/2014,NaN,52.9,249,144,104,497,...,0.780,0.398,205.0,0.0104,0.0012,4.27,2.0,3.0,3.0,5.0
3,2014,4,19/01/2014,25/01/2014,NaN,50.6,305,169,127,601,...,0.743,0.398,218.1,0.0127,0.0009,4.60,4.0,7.0,9.0,5.0
4,2014,5,26/01/2014,01/02/2014,NaN,48.3,321,171,179,671,...,1.062,0.512,254.1,NaN,0.0011,0.17,1.0,3.0,5.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658,2026,33,09/08/2026,15/08/2026,16.0,46.9,713,356,89,1190,...,1.516,0.641,149.0,0.0046,0.0014,1.44,4.0,3.0,6.0,61.0
659,2026,34,16/08/2026,22/08/2026,14.4,40.9,735,245,65,1077,...,1.476,0.588,137.5,NaN,0.0014,0.97,2.0,2.0,6.0,45.0
660,2026,35,23/08/2026,29/08/2026,17.5,40.5,846,206,60,1148,...,1.807,0.749,135.5,NaN,0.0022,1.24,0.0,8.0,4.0,54.0
661,2026,36,30/08/2026,05/09/2026,17.2,47.8,1088,181,55,1367,...,2.049,0.830,140.9,0.0057,0.0013,0.90,5.0,3.0,9.0,46.0


### For infectious disease data

In [3]:
# Locate where the datasets are stored
path = os.getcwd()
path += "\\dataset\\infectious diseases"
csv_files = glob.glob(os.path.join(path, "*.csv"))

# put all DataFrames into a list
disease_df_list = []
for f in csv_files:
    disease_df_list.append(pd.read_csv(f, usecols=[x for x in range(0, 14)]))

# Combine the DataFrames into a single one, and save it into another csv file
disease_df = pd.concat(disease_df_list, ignore_index=True)
disease_df
# data.to_csv("", index=False, encoding="utf-8-sig")


,Disease,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Total
0,Cholera,0,0,0,0,13,0,0,0,0,1,0,0,14
1,Plague,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Yellow Fever,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Acute Poliomyelitis,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Amoebic Dysentery,2,0,2,1,1,0,2,0,0,3,0,1,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1480,West Nile Virus Infection,0,0,0,0,0,0,0,0,0,0,0,0,0
1481,Whooping cough,0,1,1,0,2,0,0,0,1,1,1,0,7
1482,Yellow fever,0,0,0,0,0,0,0,0,0,0,0,0,0
1483,Zika Virus Infection,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
# Function for reading weather datas
def read_file():
    # Create an empty list for storing the DataFrames
    df_list = []

    # Read all the files and process it 
    for f in csv_files:
        temp_df = pd.read_csv(f, skiprows=2, skipfooter=3, engine="python")
        # Append the DataFrame into the list 
        df_list.append(temp_df)

    # Concat all the DataFrames into one 
    df = pd.concat(df_list, ignore_index=True)
    return df
    

### For maximum temperature

In [5]:
# Locate where the datasets are stored
path = os.getcwd()
path += "\\dataset\\daily max temp"
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Get the DataFrame with the pre-defined function
max_temp_df = read_file()
max_temp_df

# data.to_csv("", index=False, encoding="utf-8-sig")

,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1992,4,1,21.1,#
1,1992,4,2,20.3,#
2,1992,4,3,18.9,#
3,1992,4,4,23.7,C
4,1992,4,5,20.8,#
...,...,...,...,...,...
381440,2026,7,27,30.1,C
381441,2026,7,28,29.8,C
381442,2026,7,29,27.2,C
381443,2026,7,30,32.1,C


In [6]:
# Locate where the datasets are stored
path = os.getcwd()
path += "\\dataset\\daily min temp"
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Get the DataFrame with the pre-defined function
min_temp_df = read_file()
min_temp_df

# data.to_csv("", index=False, encoding="utf-8-sig")

,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1992,4,1,18.7,#
1,1992,4,2,17.4,#
2,1992,4,3,16.9,#
3,1992,4,4,18.3,C
4,1992,4,5,17.9,#
...,...,...,...,...,...
381440,2026,7,27,24.9,C
381441,2026,7,28,25.8,C
381442,2026,7,29,25.5,C
381443,2026,7,30,24.3,C


In [7]:
# Locate where the datasets are stored
path = os.getcwd()
path += "\\dataset\\daily mean temp"
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Get the DataFrame with the pre-defined function
mean_temp_df = read_file()
mean_temp_df

# data.to_csv("", index=False, encoding="utf-8-sig")

,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1992,4,1,19.3,#
1,1992,4,2,18.5,#
2,1992,4,3,17.9,C
3,1992,4,4,20.2,C
4,1992,4,5,19.2,C
...,...,...,...,...,...
382766,2026,8,27,NaN,NaN
382767,2026,8,28,NaN,NaN
382768,2026,8,29,NaN,NaN
382769,2026,8,30,NaN,NaN


In [8]:
# Locate where the datasets are stored
path = os.getcwd()
path += "\\dataset\\daily total rainfall"
csv_files = glob.glob(os.path.join(path, "*.csv"))

# Get the DataFrame with the pre-defined function
rainfall_df = read_file()
rainfall_df

# data.to_csv("", index=False, encoding="utf-8-sig")

,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1992,4.0,1.0,0.5,#
1,1992,4.0,2.0,0.5,#
2,1992,4.0,3.0,12.5,C
3,1992,4.0,4.0,17.0,C
4,1992,4.0,5.0,40.0,C
...,...,...,...,...,...
293235,2026,8.0,27.0,NaN,NaN
293236,2026,8.0,28.0,NaN,NaN
293237,2026,8.0,29.0,NaN,NaN
293238,2026,8.0,30.0,NaN,NaN


In [9]:
# Locate where the datasets are stored
path = os.getcwd()
path += "\\dataset\\daily mean humidity"
csv_files = glob.glob(os.path.join(path, "*.csv"))

humidity_df = read_file()
humidity_df
# data.to_csv("", index=False, encoding="utf-8-sig")


,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1992.0,4.0,1.0,97,#
1,1992.0,4.0,2.0,95,#
2,1992.0,4.0,3.0,96,C
3,1992.0,4.0,4.0,94,C
4,1992.0,4.0,5.0,97,C
...,...,...,...,...,...
262488,2026.0,8.0,27.0,NaN,NaN
262489,2026.0,8.0,28.0,NaN,NaN
262490,2026.0,8.0,29.0,NaN,NaN
262491,2026.0,8.0,30.0,NaN,NaN


In [11]:
max_temp_df.columns = ["Year", "Month", "Day", "Max temperature", "data Completeness"]
min_temp_df.columns = ["Year", "Month", "Day", "Min temperature", "data Completeness"]
mean_temp_df.columns = ["Year", "Month", "Day", "Mean temperature", "data Completeness"]
rainfall_df.columns = ["Year", "Month", "Day", "Rainfall", "data Completeness"]
humidity_df.columns = ["Year", "Month", "Day", "Mean humidity", "data Completeness"]


In [12]:
def data_type_conversion(df_name):
    df_name["Year"] = df_name["Year"].astype(int)
    df_name["Month"] = df_name["Month"].astype(int)
    df_name["Day"] = df_name["Day"].astype(int)
    df_name["Date"] = df_name["Year"].astype(str) + "/" + df_name["Month"].astype(str) + "/" + df_name["Day"].astype(str)
    df_name.drop(["Year", "Month", "Day"], axis=1, inplace=True)
    
    drop_index = df_name[df_name["Date"] == "1900/2/29"].index
    df_name.drop(drop_index, inplace=True)
    df_name["Date"] = pd.to_datetime(df_name["Date"], format="mixed")
    

In [13]:
data_type_conversion(max_temp_df)

In [14]:
data_type_conversion(min_temp_df)

In [15]:
data_type_conversion(mean_temp_df)


In [16]:
drop_index = humidity_df[humidity_df["Month"].isna()].index
humidity_df.drop(drop_index, inplace=True)
drop_index = rainfall_df[rainfall_df["Month"].isna()].index
rainfall_df.drop(drop_index, inplace=True)

In [17]:
data_type_conversion(rainfall_df)

In [18]:
data_type_conversion(humidity_df)


In [19]:
max_date = min(max(max_temp_df["Date"]), max(min_temp_df["Date"]), max(mean_temp_df["Date"]), max(rainfall_df["Date"]), max(humidity_df["Date"]))

def drop_rows(df_name):
    drop_index = df_name[df_name["Date"] > max_date].index
    df_name.drop(drop_index, inplace=True)


In [20]:
drop_rows(max_temp_df)
drop_rows(min_temp_df)
drop_rows(mean_temp_df)
drop_rows(rainfall_df)
drop_rows(humidity_df)

In [24]:
drop_index = max_temp_df[max_temp_df["Max temperature"] == "***"].index.to_list()
drop_index += min_temp_df[min_temp_df["Min temperature"] == "***"].index.to_list()
drop_index += mean_temp_df[mean_temp_df["Mean temperature"] == "***"].index.to_list()
drop_index += rainfall_df[rainfall_df["Rainfall"] == "***"].index.to_list()
drop_index += humidity_df[humidity_df["Mean humidity"] == "***"].index.to_list()
drop_index = set(drop_index)

max_temp_df.drop(drop_index, inplace=True)
min_temp_df.drop(drop_index, inplace=True)
mean_temp_df.drop(drop_index, inplace=True)
rainfall_df.drop(drop_index, inplace=True, errors="ignore")
humidity_df.drop(drop_index, inplace=True, errors="ignore")

In [35]:
weather_df = max_temp_df.drop("data Completeness", axis=1).merge(min_temp_df.drop("data Completeness", axis=1), how="outer", on=["Date"])
weather_df = weather_df.merge(mean_temp_df.drop("data Completeness", axis=1), how="right", on=["Date"])
weather_df = weather_df.merge(humidity_df.drop("data Completeness", axis=1), how="left", on=["Date"])
weather_df = weather_df.merge(rainfall_df.drop("data Completeness", axis=1), how="left", on=["Date"])

weather_df = weather_df.reindex(["Date", "Max temperature", "Min temperature", "Mean temperature", "Rainfall", "Mean humidity"], axis=1)

display(weather_df)
weather_df.info()


MemoryError: Unable to allocate 71.7 MiB for an array with shape (9396329,) and data type int64

In [ ]:
weather_df[["Max temperature", "Min temperature", "Mean temperature", "Mean humidity"]] = weather_df[["Max temperature", "Min temperature", "Mean temperature", "Mean humidity"]].astype(float)

pd.to_numeric(weather_df["Rainfall"], errors="coerce")

weather_df.info()

In [ ]:
weather_df[weather_df["Rainfall"] == "Trace"]

In [ ]:
weather_df.info()

In [ ]:
# ignored rainfall column **
cleaned_weather_df = weather_df.drop(["Rainfall"], axis=1).groupby(pd.Grouper(key="Date", axis=0, freq="W")).mean().round(2)
display(cleaned_weather_df)


In [ ]:
cleaned_weather_df["Year"] = pd.Series(cleaned_weather_df.index, index=cleaned_weather_df.index).dt.year
cleaned_weather_df["Week"] = pd.Series(cleaned_weather_df.index, index=cleaned_weather_df.index).dt.isocalendar().week

cleaned_weather_df.index = pd.RangeIndex(4158)
cleaned_weather_df = cleaned_weather_df.reindex(["Year", "Week", "Max temperature", "Min temperature", "Mean humidity"], axis=1)

display(cleaned_weather_df)
cleaned_weather_df.info()

In [ ]:
cleaned_weather_df[((cleaned_weather_df["Year"] == 1947) & (cleaned_weather_df["Week"] == 36))]